# Feature-relevance stability in UnimNeuron: working notebook

**Group 17, Theme 17 (Track D).** This notebook documents *what we did, how, and why*, with links to the files.
It **only reads saved results** from `results_relevance/`; it never re-runs the model.
Links are relative, so they work in VS Code, Jupyter and on GitHub (branch `mines`).

**How to refresh it:** run the experiments (see section 7), then *Run All*.

Last manual update of the text: 19 Nov 2026.

## 1. The question

> Do the learned feature-relevance weights of UnimNeuron's rule base remain stable over time on non-stationary streams, and do they shift around drift points in a way that tracks the true change in feature relevance?

| Sub-question | In short | Where it is measured |
|---|---|---|
| SQ1 stability | How much do rankings fluctuate in stationary segments, compared with a control stream? | `stab/*` in [analysis.py](../relevance/analysis.py) |
| SQ2 drift sensitivity | After drift, how many samples until the ranking is right again? | `recovery/*` |
| SQ3 validity | Do the weights rank the truly relevant features first? | `valid/*` |
| SQ4 information content | Does removing the weights (`NO_W`) hurt? | not done yet, see section 6 |

## 2. What the baseline does (read from the code, not assumed)

Source: [ENF_UnimNeuron_Safe.py](../ENF_UnimNeuron_Safe.py) and the evaluation loop in [exp_sota_comparison.py](../exp_sota_comparison.py) (`prequential_nf_generic`).

1. **Global weights `G`** (`current_feature_weights`) are recomputed every 50 samples from only the last 200 samples. They are a between-class / within-class variance ratio per feature, scaled to [0.2, 1.0].
2. **Each rule has its own weights `w`.** They are a *copy of `G` made when the rule is created* and are not refreshed afterwards.
3. **Prequential loop:** predict first, then `partial_fit`. Our runner uses the same order.
4. The model is **deterministic** (`self.rng` is never used), so seeds only change the stream.
5. With `max_rules=5` and `min_rules_for_merge=5`, **merging never fires**; the only rule removal is pruning.
6. Three kwargs passed by the baseline scripts (`safe_threshold`, `min_firing_for_update`, `drift_sensitivity`) are **silently ignored** by the model (absorbed by `**kwargs`).
7. `exp_sota_comparison.py` imports `enfs_uni0_evolving`, which is **not in the repository**. Our test stubs it (test only; the baseline is untouched).
8. The baseline's `min_max_scale` uses the whole stream (**future information**). We use an expanding min-max for real streams.

## 3. What we added, and why

The baseline is never edited. Everything is an observer next to it.

| File | What it does | Why |
|---|---|---|
| [relevance/streams.py](../relevance/streams.py) | Synthetic stream where the relevant features switch at a known time, plus the truth matrix; leak-free scaling; shuffled control | We need ground truth. River's Agrawal/SEA drifts do not necessarily change *which* features matter |
| [relevance/extract.py](../relevance/extract.py) | Read-only snapshots of `G`, of `A` (support-weighted mean of rule weights) and of per-rule weights; rule ID tracker | The proposal promises non-intrusive logging. Rules have no IDs in the model |
| [relevance/logger.py](../relevance/logger.py) | Saves a checkpoint every 50 samples (matches the model's own refresh period) | Weights over time are not stored by the baseline |
| [relevance/metrics.py](../relevance/metrics.py) | Kendall tau-b, top-k Jaccard, top-k precision (tie-aware), AUC, sustained-recovery helper | One tool per sub-question |
| [relevance/analysis.py](../relevance/analysis.py) | Stability at several lags, validity per segment, ranking-recovery time, accuracy recovery | Snapshots 50 apart share 75% of their data, so we also use lags of 200, 500, 2000 |
| [run_relevance.py](../run_relevance.py) | Config- and seed-driven prequential run with logging | Reproducibility |
| [configs/mve.json](../configs/mve.json), [configs/stationary_control.json](../configs/stationary_control.json) | Protocol parameters; the control has no drift | The control gives the noise floor for SQ1 |
| [tests/test_nonintrusive.py](../tests/test_nonintrusive.py) | Non-intrusiveness, no-label-leakage, metric sanity, equality with the baseline loop | Promises made in the proposal |

## 4. Verification

Run on 19 Nov 2026 (`python tests/test_nonintrusive.py`):

- logging does not change predictions, rule counts, weights or rule state: **PASS**
- changing a label at time *t* does not change predictions up to *t* (no leakage): **PASS**
- on the same stream, our loop gives **identical predictions and rule counts** to the baseline `prequential_nf_generic`: **PASS** (with `enfs_uni0_evolving` stubbed in the test)
- metric sanity checks: **PASS**

Logging cost: about 2% of run time (`log_time_s` against `proc_time_s` in the results below).

In [ ]:
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

ROOT = Path.cwd()
if not (ROOT / "relevance").exists():
    ROOT = ROOT.parent
RES = ROOT / "results_relevance"

def load_summary(name):
    p = RES / name / "summary.json"
    return json.load(open(p)) if p.exists() else None

def load_seed(name, seed):
    p = RES / name / f"seed_{seed}" / "checkpoints.npz"
    return dict(np.load(p)) if p.exists() else None

def fmt(s, key, nd=3):
    v = s["summary"].get(key) if s else None
    if v is None or v["mean"] is None:
        return "n/a"
    return f'{v["mean"]:.{nd}f} ± {v["std"]:.{nd}f}'

mve, ctl = load_summary("mve"), load_summary("stationary_control")
print("repo root:", ROOT)
print("mve results found:", mve is not None, "| control results found:", ctl is not None)
if mve is None:
    print("Run first:  python run_relevance.py --config configs/mve.json --seeds 0 1 2 --plot")

## 5. Results so far

All numbers: mean ± std over the seeds found in `results_relevance/`. The model is deterministic, so the spread comes from the stream generator only.

### 5.1 Raw weight trajectories (first thing to look at)

Blue = features relevant before the drift, red = relevant after, grey = never relevant. Dashed line = drift point.

In [ ]:
if mve is not None:
    d = load_seed("mve", 0)
    cfg = mve["config"]
    rb, ra = set(cfg["stream"]["rel_before"]), set(cfg["stream"]["rel_after"])
    drift = cfg["stream"]["drift_at"]
    col = lambda j: "tab:blue" if j in rb else "tab:red" if j in ra else "lightgray"
    t = d["t"]
    fig, ax = plt.subplots(3, 1, figsize=(8, 8), sharex=True)
    for j in range(d["G"].shape[1]):
        ax[0].plot(t, d["G"][:, j], color=col(j), lw=1)
        ax[1].plot(t, d["A"][:, j], color=col(j), lw=1)
    ax[0].set_title("G: global weights (current_feature_weights)")
    ax[1].set_title("A: rule-base weights (support-weighted mean of rule.w)")
    ax[2].plot(t, d["acc_win"], label="windowed accuracy")
    ax[2].set_ylabel("accuracy"); ax[2].set_ylim(0, 1)
    r = ax[2].twinx(); r.plot(t, d["n_rules"], color="k", lw=0.8); r.set_ylabel("number of rules")
    for a in ax:
        a.axvline(drift, color="red", ls="--", lw=0.8)
    ax[2].set_xlabel("samples")
    fig.tight_layout(); plt.show()

**Reading it:** `G` switches from the blue to the red features after the drift. `A` stays flat: the rules were created before sample ~20 and their weights never change. Accuracy stays near 0.5.

### 5.2 Stability (SQ1): drift stream against the stationary control

In [ ]:
if mve is not None and ctl is not None:
    lags = mve["config"]["analysis"]["lags_samples"]
    rows = ["| lag (samples) | G tau-b, drift stream | G tau-b, control | G top-3 Jaccard, drift stream | G top-3 Jaccard, control |",
            "|---|---|---|---|---|"]
    for L in lags:
        rows.append(f'| {L} | {fmt(mve, f"stab/G/tau_b/lag{L}")} | {fmt(ctl, f"stab/G/tau_b/lag{L}")} | '
                    f'{fmt(mve, f"stab/G/jaccard/lag{L}")} | {fmt(ctl, f"stab/G/jaccard/lag{L}")} |')
    display(Markdown("\n".join(rows)))

**Reading it:** the top-3 set never changes (Jaccard = 1). Full-ranking tau-b plateaus near 0.47 from lag 200 and is the same in the control, so that plateau is a **noise floor**: the 7 irrelevant features all sit near the 0.2 weight floor and their order is random. Lag 50 is higher only because snapshots overlap by 75%.

### 5.3 Validity (SQ3) and drift recovery (SQ2)

In [ ]:
if mve is not None:
    k = mve["config"]["analysis"]["topk"]
    rows = ["| measure | G (global) | A (rule base) |", "|---|---|---|"]
    for lab, key in [(f"top-{k} precision, before drift", "valid/{}/prec_at_k/seg0"),
                     (f"top-{k} precision, after drift", "valid/{}/prec_at_k/seg1"),
                     ("AUC, before drift", "valid/{}/auc/seg0"),
                     ("AUC, after drift", "valid/{}/auc/seg1"),
                     ("ranking recovery time (samples after drift)", "recovery/{}/drift1"),
                     ("variance of weights, before drift", "var/{}/seg0"),
                     ("share of checkpoints with all weights equal", "frac_constant/{}")]:
        rows.append(f"| {lab} | {fmt(mve, key.format('G'))} | {fmt(mve, key.format('A'))} |")
    display(Markdown("\n".join(rows)))

**Reading it:** `G` identifies the relevant features perfectly in both segments and re-orients about 150 samples after the drift, which is close to the floor imposed by its 200-sample buffer. `A` does not follow the drift, and in some seeds it is uniform for the whole run, so its before-drift numbers depend on the seed.

### 5.4 Predictive and resource results

In [ ]:
if mve is not None:
    rows = ["| measure | drift stream | stationary control |", "|---|---|---|"]
    for lab, key, nd in [("final accuracy", "acc_final", 3), ("final number of rules", "rules_final", 1),
                         ("rules created (total)", "struct/created_total", 1), ("rules removed (total)", "struct/removed_total", 1),
                         ("processing time [s]", "proc_time_s", 1), ("logging time [s]", "log_time_s", 2)]:
        rows.append(f"| {lab} | {fmt(mve, key, nd)} | {fmt(ctl, key, nd)} |")
    display(Markdown("\n".join(rows)))

**Do not report accuracy-recovery numbers from this stream.** Accuracy is at chance (0.50), so `acc/drop` and `acc/recovery` in `metrics.json` are noise in a 200-sample window, not a real drop and recovery.

## 6. Findings log and open problems

**Established by runs (19 Nov 2026, our synthetic stream, baseline config, 3 seeds):**

1. `G` tracks the ground truth (top-3 precision 1.0 and AUC 1.0 in both segments) and re-orients about 150 samples after an abrupt drift.
2. The top-k set is stable; the full ranking is not, because of noise among irrelevant features. The choice of metric changes the conclusion for H1.
3. `A` (what the rules actually use) is frozen: all surviving rules were created in the first ~20 samples with uniform weights (`1/d`), before `G` is first refreshed at sample 50.
4. Accuracy is about 0.50 on our streams, **also on separable cluster streams we tried (d=10 and d=4)** with the baseline config. We do not yet know why.
5. In the FULL model, rules born before sample 50 have effective weight `1/d`, which squashes their activations toward the neutral element `e`. The ablation's `NO_W` ([exp_ablation.py](../exp_ablation.py)) sets all weights to 1 after every `partial_fit`, so it changes the **scale** of the weights as well as their information content.

**Not known yet:**

- A stream where the model is clearly above chance (needed for accuracy claims and for SQ4). Check `results_exp/exp_stream_summary.tex` and the Agrawal / Elec2 / Phishing runs.
- Whether rules are still created after sample 50 on those streams (that decides whether `A` can ever carry learned relevance).
- Behaviour on a real stream (Elec2, with a shuffled control).

**Proposed changes to the protocol:**

- Use top-k Jaccard / precision as the primary stability metrics; report tau-b next to the control as a noise floor.
- Reformulate H3: compare `G` with `A` and measure how far `A` lags `G`, instead of per-rule stability (per-rule weights are frozen copies).
- For SQ4, add a **shuffled-weights control** (permute the weights across features, keeping their scale) next to `NO_W`.

**Next steps**

- Pick the stream(s) where the model is above chance
- Freeze `configs/mve.json` (`topk`, `recovery_consecutive`, `warmup_flush`, `acc_tol`) and commit before looking at more results
- Run a real stream (Elec2) and its shuffled control
- Decide SQ4 design
- Add entries to `CONTRIBUTIONS.md`

## 7. How to reproduce

```powershell
python -m venv .venv
.\.venv\Scripts\Activate.ps1
python -m pip install -r requirements.txt
python tests/test_nonintrusive.py
python run_relevance.py --config configs/mve.json --seeds 0 1 2 --plot
python run_relevance.py --config configs/stationary_control.json --seeds 0 1 2
```

Per-seed outputs are in `results_relevance/<config name>/seed_<n>/`: `checkpoints.npz`, `per_rule.csv`, `metrics.json`, `trajectories.png`. The aggregate over seeds is in `summary.json`.